# GovBench-Med — Full Experiment Run (Google Colab)

**Purpose**: Run the complete 300-case × 5-level × 3-model × 3-seed matrix on Colab T4 GPU (~3-5s/call).

## Setup
1. Open this notebook in Colab: `File → Open notebook → GitHub` or upload `.ipynb`
2. **Runtime → Change runtime type → T4 GPU** (free tier)
3. Run cells in order
4. Download results when done: `experiments/results/*.csv` + `experiments/logs/*.json`


In [ ]:
# =====================================================================
# CELL 1: Install Ollama on Colab (no GPU mode needed for llama.cpp)
# =====================================================================
!curl -fsSL https://ollama.com/install.sh | sh
# Start Ollama server in background
import subprocess, time, threading, sys

def run_ollama():
    subprocess.run(['ollama', 'serve'], capture_output=True)

ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()
time.sleep(3)
print('Ollama server started')

# Pull models (T4 GPU has 16GB VRAM - fits 8B models easily)
for model in ['llama3.1:8b', 'mistral:7b', 'qwen2.5:7b']:
    print(f'Pulling {model}...')
    result = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True, timeout=600)
    if result.returncode == 0:
        print(f'  ✓ {model}')
    else:
        print(f'  ✗ {model}: {result.stderr[:200]}')


In [ ]:
# =====================================================================
# CELL 2: Clone the project repo
# =====================================================================
import os, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/govbench-med.git"  # <-- UPDATE THIS

if not os.path.exists('govbench-med'):
    subprocess.run(['git', 'clone', REPO_URL])
else:
    subprocess.run(['git', '-C', 'govbench-med', 'pull'])

os.chdir('govbench-med')
print('Working in:', os.getcwd())

In [ ]:
# =====================================================================
# CELL 3: Install Python deps (fast on Colab)
# =====================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'requests', 'transformers', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'scipy', 'scikit-learn', 'tqdm'])
print('Dependencies installed')

In [ ]:
# =====================================================================
# CELL 4: Prepare data (downloads real MedQA + DDXPlus)
# =====================================================================
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/prepare_data.py'],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# =====================================================================
# CELL 5: Verify single call speed (should be 3-5s on T4)
# =====================================================================
import time, requests, json

t0 = time.time()
r = requests.post('http://localhost:11434/api/generate',
    json={'model': 'llama3.1:8b', 'prompt': 'Say hi', 'stream': False}, timeout=30)
print(f'Latency: {time.time()-t0:.1f}s')
print(f'Response: {r.json().get("response", "")[:50]}')

In [ ]:
# =====================================================================
# CELL 6: Run the FULL experiment matrix
# =====================================================================
# This runs: 300 cases × 5 levels × 3 models × 3 seeds = 13,500 runs
# At ~4s/call on T4, expect 10-15 hours total.
# You can run subsets with flags:
#   --n 50 --level G0 G1 G2 --model llama3.1:8b --seed 42

import subprocess, sys

# FULL RUN (uncomment to run everything):
# subprocess.run([sys.executable, 'scripts/run_experiments.py', '--full'])

# QUICK RUN: 50 cases × G0,G1,G2 × 1 model × 1 seed = 150 runs (~10 min)
subprocess.run([sys.executable, 'scripts/run_experiments.py',
    '--n', '50',
    '--level', 'G0',
    '--level', 'G1',
    '--level', 'G2',
    '--model', 'llama3.1:8b',
    '--seed', '42'])

print('Quick run complete. Check experiments/results/')

In [ ]:
# =====================================================================
# CELL 7: Generate Pareto curves and paper figures
# =====================================================================
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import json, glob, os

# Load latest results CSV
csvs = sorted(glob.glob('experiments/results/results_*.csv'))
if csvs:
    df = pd.read_csv(csvs[-1])
    print(f'Loaded {len(df)} rows from {csvs[-1]}')
    print(df.head())
    
    # Aggregate by (model, governance_level)
    agg = df.groupby(['model', 'governance_level']).agg({
        'css': 'mean',
        'cmr': 'mean',
        'hir': 'mean',
        'urr': 'mean',
        'total_tokens': 'mean',
        'total_latency': 'mean',
        'top1_correct': 'mean',
    }).reset_index()
    
    # Plot CSS vs CCS (Pareto frontier)
    agg['ccs'] = 0.6 * (agg['total_tokens'] / agg[agg['governance_level']=='G0']['total_tokens'].values[0]) \
               + 0.4 * (agg['total_latency'] / agg[agg['governance_level']=='G0']['total_latency'].values[0])
    
    plt.figure(figsize=(10, 6))
    for model in agg['model'].unique():
        mdf = agg[agg['model'] == model].sort_values('governance_level')
        plt.plot(mdf['ccs'], mdf['css'], 'o-', label=model, linewidth=2, markersize=8)
        for _, row in mdf.iterrows():
            plt.annotate(row['governance_level'], (row['ccs'], row['css']),
                        textcoords='offset points', xytext=(5, 5))
    plt.xlabel('Composite Cost Score (CCS)')
    plt.ylabel('Clinical Safety Score (CSS)')
    plt.title('GovBench-Med: Governance-Cost Pareto Frontier')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('paper/figures/pareto_frontier.png', dpi=300)
    plt.show()
    
    print('\nAggregate metrics:')
    print(agg.to_string(index=False))
else:
    print('No results CSV found. Run experiments first.')

In [ ]:
# =====================================================================
# CELL 8: Download results to your computer
# =====================================================================
from google.colab import files
import glob, zipfile, os

result_files = glob.glob('experiments/results/*.csv') + glob.glob('experiments/results/*.json')
log_files = glob.glob('experiments/logs/*.json')[:50]  # sample

with zipfile.ZipFile('govbench_results.zip', 'w') as zf:
    for f in result_files:
        zf.write(f)
    for f in log_files:
        zf.write(f)

print(f'Packed {len(result_files)} result files + {len(log_files)} logs')
files.download('govbench_results.zip')